In [1]:
import os
import sys

import numpy as np
import torch
import torch.nn as nn

from lib.dataset.const import INITIAL_JOINT_ANGLE
from lib.models.backbones.Resnet import get_resnet
from lib.models.backbones.HRnet import get_hrnet
from lib.utils.utils import set_random_seed, create_logger, get_dataloaders, get_scheduler, resume_run, save_checkpoint
from lib.utils.urdf_robot import URDFRobot
from easydict import EasyDict

torch.cuda.set_device(5)
args_for_data = EasyDict({
    "urdf_robot_name": "panda",
    "train_ds_names": "./data/dream/real/panda-orb",
    "val_ds_names": None,
    "image_size": 256.0,

    "jitter": True,
    "other_aug": True,
    "occlusion": True,
    "occlu_p": 0.5,
    "padding": False,
    "fix_truncation": False,
    "truncation_padding": [120, 120, 120, 120],
    "rootnet_flip": False,


    "backbone_name": "resnet50",
    "rootnet_backbone_name": "hrnet32",
    "rootnet_image_size": 256.0,
    "other_image_size": 256.0,
    "use_rpmg": False,
    "batch_size": 48,
    "epoch_size": 104950,
    "n_epochs": 700,
    "n_dataloader_workers": 6,
    "save_epoch_interval": None,
    "clip_gradient": 5.0})

robot = URDFRobot(args_for_data.urdf_robot_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ds_iter_train, test_loader_dict = get_dataloaders(args_for_data)

init_param_dict = {
        "robot_type" : args_for_data.urdf_robot_name,
        "pose_params": INITIAL_JOINT_ANGLE,
        "cam_params": np.eye(4,dtype=float),
        "init_pose_from_mean": True
    }

/home/ruihengwang/robot_pose_estim/Holistic-Robot-Pose-Estimation/lib/utils/urdfpytorch/urdf.py:2169: RuntimeWarning: invalid value encountered in divide
  value = value / np.linalg.norm(value)
32315it [00:00, 146358.33it/s]
5997it [00:00, 175533.45it/s]
5999it [00:00, 174370.27it/s]
6394it [00:00, 174233.57it/s]
4966it [00:00, 177645.32it/s]
5944it [00:00, 178283.19it/s]
32315it [00:00, 172609.83it/s]


len(ds_iter_train):  674
len(ds_iter_test_dr):  125
len(ds_iter_test_photo):  125
len(ds_iter_test_azure):  134
len(ds_iter_test_kinect):  104
len(ds_iter_test_realsense):  124
len(ds_iter_test_orb):  674


In [3]:
model_backbone = get_resnet(args_for_data.backbone_name)
depth_backbone = get_hrnet(type_name=32, num_joints=7, depth_dim=64,
                            pretrain=True, generate_feat=True, generate_hm=False)
# model_backbone
from tqdm import tqdm
from torchnet.meter import AverageValueMeter
from lib.utils.utils import cast
from lib.utils.integral import HeatmapIntegralJoint, HeatmapIntegralPose
from lib.models.class_head import ClassificationHead
from lib.models.tokenizer import VectorQuantizeTokenizer
args_for_tokenizer = EasyDict({
    "encoder_num_blocks": 3,
    "num_joints": 7,
    "encoder_num_blocks": 4,
    "encoder_token_inter_dim": 64,
    "encoder_hidden_dim": 128,
    "encoder_hidden_inter_dim": 32,
    "encoder_dropout": 0.1,
    "token_num": 128,
    "token_class_num": 1024,
    "token_dim": 128,
    "ema_decay": 0.99,
    "decoder_num_blocks": 4,
    "decoder_hidden_dim": 128,
    "decoder_hidden_inter_dim": 32,
    "decoder_token_inter_dim": 64,
    "decoder_p_dropout": 0.1,
    "tokenizer_pretrained": "./experiments/panda_full_w_vq_3d_synth_dr/tokenizer_pretrained/epoch_50_tokenizer.pk"
})
integral_layer = HeatmapIntegralPose(backbone=args_for_data.backbone_name,
                                     num_joints=7,
                                     depth_dim=64,
                                     height_dim=64,
                                     width_dim=64,
                                     norm_type="softmax",
                                     image_size=256.0,
                                     bbox_3d_shape=[1300, 1300, 1300],
                                     root_id=3,
                                     fixroot=True)

tokenizer = VectorQuantizeTokenizer(input_dim=3,
                                    output_dim=3,
                                    encoder_num_blocks=args_for_tokenizer.encoder_num_blocks,
                                    num_joints=args_for_tokenizer.num_joints,
                                    encoder_token_inter_dim=args_for_tokenizer.encoder_token_inter_dim,
                                    encoder_hidden_dim=args_for_tokenizer.encoder_hidden_dim,
                                    encoder_hidden_inter_dim=args_for_tokenizer.encoder_hidden_inter_dim,
                                    encoder_dropout=args_for_tokenizer.encoder_dropout,
                                    token_num=args_for_tokenizer.token_num,
                                    token_class_num=args_for_tokenizer.token_class_num,
                                    token_dim=args_for_tokenizer.token_dim,
                                    ema_decay=args_for_tokenizer.ema_decay,
                                    decoder_num_blocks=args_for_tokenizer.decoder_num_blocks,
                                    decoder_hidden_dim=args_for_tokenizer.decoder_hidden_dim,
                                    decoder_hidden_inter_dim=args_for_tokenizer.decoder_hidden_inter_dim,
                                    decoder_token_inter_dim=args_for_tokenizer.decoder_token_inter_dim,
                                    decoder_p_dropout=args_for_tokenizer.decoder_p_dropout,
                                    stage="classifier")
tokenizer.init_weights(pretrained=args_for_tokenizer.tokenizer_pretrained)




/home/ruihengwang/miniconda3/envs/hoilistic/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/ruihengwang/miniconda3/envs/hoilistic/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Initialized resnet50 from model zoo
Loading hrnet pretrained weights (ImageNet) from ./models/hrnet_w32-36af842e_roc.pth
Loading pretrained tokenizer....
Tokenizer pretrain weight loaded successfully!


In [4]:
args_for_class_head = EasyDict(
{
    "class_in_channels": 2048,
    "class_hidden_dim": 128,
    "class_num_blocks": 3,
    "class_hidden_inter_dim": 64,
    "class_token_inter_dim": 64,
    "class_conv_channels": 128,
    "class_p_dropout": 0.1
})
class_head = ClassificationHead(
    in_channels=args_for_class_head.class_in_channels, 
    image_size=(256.0, 256.0),
    num_joints=args_for_tokenizer.num_joints,
    conv_channels=args_for_class_head.class_conv_channels,
    hidden_dim=args_for_class_head.class_hidden_dim,
    num_blocks=args_for_class_head.class_num_blocks,
    hidden_inter_dim=args_for_class_head.class_hidden_inter_dim,
    token_inter_dim=args_for_class_head.class_token_inter_dim,
    dropout=args_for_class_head.class_p_dropout,
    token_num=args_for_tokenizer.token_num,
    token_class_num=args_for_tokenizer.token_class_num,
    tokenizer=tokenizer
)


In [5]:
torch.cuda.set_device(5)
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.tensorboard import SummaryWriter
import torch.nn.functional as F

log_dir = "new_exp/vq_logdir_trainon_realorb"
os.makedirs(log_dir, exist_ok=True)

tb_writer = SummaryWriter(log_dir=os.path.join(log_dir, "tensorboard"))
log_csv = os.path.join(log_dir, "training_loss_vq.csv")
columns = ["epoch", "step", "loss_total", "loss_3dkp", "loss_token"]
df = pd.DataFrame(columns=columns)
losses = []
class myModelVQ(nn.Module):

    def __init__(self, model_backbone, depth_backbone, class_head):
        super().__init__()
        self.model_backbone = model_backbone
        self.depth_backbone = depth_backbone
        self.class_head = class_head

    def forward(self, x_reg_input, x_root_input, **kwargs):
        batch_size = x_reg_input.shape[0]
        x_reg_input = x_reg_input.to(torch.float)
        x_root_input = x_root_input.to(torch.float)

        x_out = self.model_backbone(x_reg_input)
        gt_keypoints3d = kwargs.get("gt_keypoints3d", None)
        is_training = kwargs.get("train", True)
        if is_training:
            pred_logit, pred_xyz_int, gt_indicies = self.class_head(x_out, joints=gt_keypoints3d, train=is_training)
            return pred_logit, pred_xyz_int, gt_indicies
        else: 
            pred_xyz_int, _ = self.class_head(x_out, joints=gt_keypoints3d, train=is_training)
            return pred_xyz_int
    
model = myModelVQ(model_backbone, depth_backbone, class_head)

args_for_optim = EasyDict({
    "lr": 1e-4,
    "weight_decay": 0.0,
    "use_schedule": True,
    "schedule_type": "exponential",
    "n_epochs_warmup": 0,
    "start_decay": 45,
    "end_decay": 100,
    "final_decay": 0.01,
    "exponent": 0.95,
    "n_epochs": 20,
    "batch_size": 48,
    "clip_gradient": 5.0
})
optimizer = torch.optim.Adam(model.parameters(), lr=args_for_optim.lr, weight_decay=args_for_optim.weight_decay)
curr_max_auc = 0.0
curr_max_auc_4real = { "azure": 0.0, "kinect": 0.0, "realsense": 0.0, "orb": 0.0 }
start_epoch, last_epoch, end_epoch = 0, -1, args_for_optim.n_epochs
lr_scheduler = get_scheduler(args_for_optim, optimizer, last_epoch)

for epoch in range(start_epoch, end_epoch + 1):
    model.train()
    model.to(device)
    print("Epoch: {}".format(epoch))
    iterator = tqdm(ds_iter_train, dynamic_ncols=True)
    losses_3dkp = AverageValueMeter()  # 使用 torchnet 的 AverageValueMeter
    losses_ce_token = AverageValueMeter()
    losses_total = AverageValueMeter()

    for batch_idx, input_batch in enumerate(iterator):
        optimizer.zero_grad()
        root_images = cast(input_batch["root"]["images"], device).float() / 255.
        root_K = cast(input_batch["root"]["K"], device).float()
        reg_images = cast(input_batch["other"]["images"], device).float() / 255.
        other_K = cast(input_batch["other"]["K"], device).float()
        bboxes = cast(input_batch["root"]["bbox_gt2d_extended"], device).float()

        batch_size = reg_images.shape[0]
        gt_keypoints3d = cast(input_batch["other"]["keypoints_3d"], device).float()
        gt_keypoints2d = cast(input_batch["other"]["keypoints_2d"], device).float()

        real_bbox = torch.tensor([1000.0, 1000.0]).to(torch.float32)
        fx, fy = root_K[:, 0, 0], root_K[:, 1, 1]
        area = torch.max(torch.abs(bboxes[:, 2] - bboxes[:, 0]), torch.abs(bboxes[:, 3] - bboxes[:, 1])) ** 2
        k_values = torch.tensor([torch.sqrt(fx[n] * fy[n] * real_bbox[0] * real_bbox[1] / area[n]) for n in range(batch_size)]).to(torch.float32)
        k_values = cast(k_values, device)

        pred_logit, pred_keypoints3d_int, gt_indicies = model(reg_images, reg_images, gt_keypoints3d=gt_keypoints3d)
        error3d_int = torch.norm(pred_keypoints3d_int - gt_keypoints3d, dim=2)
        error3d_int = cast(error3d_int, device)
        

        loss_error3d_int = torch.mean(error3d_int)
        ce_loss = F.cross_entropy(pred_logit, gt_indicies)

        loss = ce_loss + loss_error3d_int
        loss.backward()
        optimizer.step()

        # 更新损失值到 AverageValueMeter
        losses_3dkp.add(loss_error3d_int.item())
        losses_ce_token.add(ce_loss.item())
        losses_total.add(loss.item())

        # 每隔 100 个 batch 记录一次损失值到 TensorBoard，并重置 AverageValueMeter
        if (batch_idx + 1) % 100 == 0:
            tb_writer.add_scalar("Loss/total", losses_total.mean, epoch * len(iterator) + batch_idx + 1)
            tb_writer.add_scalar("Loss/3d_error", losses_3dkp.mean, epoch * len(iterator) + batch_idx + 1)
            tb_writer.add_scalar("Loss/tokens", losses_ce_token.mean, epoch * len(iterator) + batch_idx + 1)
            new_row = {
                "epoch": epoch,
                "step": batch_idx + 1,
                "loss_total": losses_total.mean,
                "loss_3dkp": losses_3dkp.mean,
                "loss_token": losses_ce_token.mean
            }
            df = df.append(new_row, ignore_index=True)
            losses_3dkp.reset()  
            losses_ce_token.reset()
            losses_total.reset()
            
    # 每个 epoch 结束时，记录该 epoch 的平均损失值到日志文件
    
    df.to_csv(log_csv, index=False)

    # 更新学习率调度器
    lr_scheduler.step()


tb_writer.close()

Epoch: 0


 15%|█▍        | 99/674 [00:46<02:47,  3.44it/s] /tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 30%|██▉       | 199/674 [01:20<02:20,  3.39it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 44%|████▍     | 299/674 [01:53<02:11,  2.85it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 59%|█████▉    | 399/674 [02:26<02:08,  2.15it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future versi

Epoch: 1


 15%|█▍        | 99/674 [00:21<01:44,  5.51it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 30%|██▉       | 199/674 [00:40<01:21,  5.80it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 44%|████▍     | 299/674 [00:58<01:04,  5.78it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 59%|█████▉    | 399/674 [01:16<00:45,  6.06it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future versio

Epoch: 2


...

 15%|█▍        | 99/674 [00:22<01:57,  4.88it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 30%|██▉       | 199/674 [00:43<01:40,  4.71it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 44%|████▍     | 299/674 [01:05<01:20,  4.63it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 59%|█████▉    | 399/674 [01:27<01:02,  4.37it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future versio

Epoch: 19


 15%|█▍        | 99/674 [00:21<02:03,  4.67it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 30%|██▉       | 199/674 [00:43<01:41,  4.68it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 44%|████▍     | 299/674 [01:05<01:15,  4.98it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 59%|█████▉    | 399/674 [01:26<00:52,  5.29it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future versio

Epoch: 20


 15%|█▍        | 99/674 [00:20<01:57,  4.90it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 30%|██▉       | 199/674 [00:41<01:46,  4.45it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 44%|████▍     | 299/674 [01:03<01:19,  4.72it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 59%|█████▉    | 399/674 [01:25<01:04,  4.28it/s]/tmp/ipykernel_2644823/2680282528.py:118: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future versio